In [8]:
import json
import pandas as pd
import yaml

In [11]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

def flatten_dict(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

In [5]:
with open('../configs/personas_v2.yaml', 'r') as file:
    personas = yaml.safe_load(file)

In [57]:
persona_df = pd.DataFrame(personas['law_enforcement']['personas'])[['Name','uuid']]

In [34]:
sjt_questions = read_json("sjt_data/synthetic_generated_sjt_list.json")
sjt_questions_v2 = read_json("sjt_data/synthetic_generated_sjt_list_v2.json")

sjt_questions = sjt_questions + sjt_questions_v2

sjt_questions_df = pd.DataFrame(sjt_questions)

In [6]:
sjt_answers = read_json("sjt_answers_handmade_personas_62_sjts.json")

flatten_sjt_answers = [flatten_dict(answer) for answer in sjt_answers]

In [22]:
persona_id_list = []
for sjt_answer in flatten_sjt_answers:
    persona_id_list.extend([sjt_answer['config.persona_id']]*62)

flattend_question_hashes = pd.DataFrame(flatten_sjt_answers)['config.question_hashes'].explode()
flattend_answers = pd.DataFrame(flatten_sjt_answers)['answers'].explode()

In [38]:
answer_df = pd.DataFrame({"Persona_id": persona_id_list,
              "question_id": flattend_question_hashes,
              "answers": flattend_answers})

In [48]:
final_answer_df = pd.merge(answer_df, sjt_questions_df[['hash_id', 'question',
       'honesty_humility_option', 'emotionality_option', 'extraversion_option',
       'agreeableness_option', 'conscientiousness_option', 'openness_option']], left_on = "question_id", right_on = "hash_id")

final_answer_df = pd.merge(final_answer_df, persona_df, left_on = "Persona_id", right_on = "uuid")

In [62]:
final_answer_df.to_csv("sjt_data/final_answer_df_handmade_personas_62_sjts.csv", index = False)